# ETL
This is a one-time script that loads and converts the csv files into a single parquet file for better storage and future use.

## More context
- I first extracted the downloaded zip file. There were in total 365 csv files (one for each day).
- In this script, I combined all the csv and converted into parquet format.
- Data source: https://tdx.transportdata.tw/

In [ ]:
import polars as pl
from constants import DATA_FOLDER, DATA_FILE

print(DATA_FOLDER)

c:\Code\Python\Projects\04-BusTime\raw_data


In [ ]:
# At the time, there were 365 csv files sitting in the data folder. (and only the csv files, nothing else)
files = [f.name for f in DATA_FOLDER.iterdir() if f.is_file()]
cols = pl.scan_csv(DATA_FOLDER / files[0]).collect_schema().names()
print(cols)
print(len(cols))

In [4]:
for file in files:
    if cols != pl.scan_csv(DATA_FOLDER / file).collect_schema().names():
        print(f"{file} has a different header.")

Every csv has the same header.

In [5]:
df = pl.read_csv(DATA_FOLDER / files[1], ignore_errors=True)
df.glimpse()

Rows: 529710
Columns: 29
$ PlateNumb         <str> 'KKA-3975', 'KKA-3975', 'KKA-3975', 'KKA-3975', 'KKA-3975', '359-FQ', '359-FQ', 'KKA-3975', 'KKA-3975', 'KKA-3975'
$ OperatorID        <i64> 16, 16, 16, 16, 16, 35, 35, 16, 16, 16
$ OperatorNo        <i64> 702, 702, 702, 702, 702, 1308, 1308, 702, 702, 702
$ RouteUID          <str> 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968', 'THB0968'
$ RouteID           <i64> 968, 968, 968, 968, 968, 968, 968, 968, 968, 968
$ RouteNameZh_tw    <i64> 968, 968, 968, 968, 968, 968, 968, 968, 968, 968
$ RouteNameEn       <i64> 968, 968, 968, 968, 968, 968, 968, 968, 968, 968
$ SubRouteUID       <str> 'THB096802', 'THB096802', 'THB096802', 'THB096802', 'THB096801', 'THB096801', 'THB096801', 'THB096802', 'THB096802', 'THB096802'
$ SubRouteID        <i64> 96802, 96802, 96802, 96802, 96801, 96801, 96801, 96802, 96802, 96802
$ SubRouteNameZh_tw <i64> 968, 968, 968, 968, 968, 968, 968, 968, 968, 968
$ SubRo

In [6]:
schema = {
    "PlateNumb": pl.String,
    "OperatorID": pl.Int32,
    "OperatorNo": pl.Int32,
    "RouteUID": pl.String,
    "RouteID": pl.Int32,
    "RouteNameZh_tw": pl.String,
    "RouteNameEn": pl.String,
    "SubRouteUID": pl.String,
    "SubRouteID": pl.String,
    "SubRouteNameZh_tw": pl.String,
    "SubRouteNameEn": pl.String,
    "Direction": pl.Categorical,
    "StopUID": pl.String,
    "StopID": pl.Int32,
    "StopNameZh_tw": pl.String,
    "StopNameEn": pl.String,
    "StopSequence": pl.Int32,
    "MessageType": pl.Categorical,
    "DutyStatus": pl.Categorical,
    "BusStatus": pl.Categorical,
    "A2EventType": pl.Categorical,
    "GPSTime": pl.String,
    "TripStartTimeType": pl.Categorical,
    "TripStartTime": pl.String,
    "TransTime": pl.String,
    "SrcRecTime": pl.String,
    "SrcTransTime": pl.String,
    "SrcUpdateTime": pl.String,
    "UpdateTime": pl.String,
}

In [ ]:
# Identify the corrupted csv files

error_files = []
for file in files:
    try:
        df = (
            pl.scan_csv(
                DATA_FOLDER / file,
                schema=schema,
            )
            .null_count()
            .collect()
        )
    except Exception as e:
        error_files.append(file)

print(error_files)

['公路客運定點歷史資料(A2)2025-03-08.CSV', '公路客運定點歷史資料(A2)2025-04-05.CSV', '公路客運定點歷史資料(A2)2025-05-12.CSV', '公路客運定點歷史資料(A2)2025-05-21.CSV', '公路客運定點歷史資料(A2)2025-06-03.CSV', '公路客運定點歷史資料(A2)2025-06-07.CSV', '公路客運定點歷史資料(A2)2025-06-12.CSV']


In [ ]:
# Convert the correct files to parquet format for more efficient data storage and future use
correct_files = [DATA_FOLDER / file for file in files if file not in error_files]

pl.scan_csv(
    correct_files,
    schema=schema,
).sink_parquet(DATA_FILE)

In [ ]:
# Check/Load the parquet file
df = pl.scan_parquet(DATA_FILE)
target = (
    df.drop(
        [
            "OperatorID",
            "OperatorNo",
            "RouteUID",
            "OperatorID",
            "OperatorNo",
            "RouteUID",
            "StopUID",
            "StopID",
            "TripStartTime",
            "TransTime",
            "SrcRecTime",
            "SrcTransTime",
            "SrcUpdateTime",
            "UpdateTime",
            "RouteNameZh_tw",
            "RouteNameEn",
            "SubRouteUID",
            "SubRouteID",
            "SubRouteNameZh_tw",
            "SubRouteNameEn",
            "StopNameEn",
            "MessageType",
            "TripStartTimeType",
        ]
    )
    .filter(
        pl.col("RouteID") == 1728,
        pl.col("StopNameZh_tw").is_in(["新竹轉運站", "龍潭運動公園"]),
        pl.col("A2EventType") == "離站",
    )
    .collect()
)

In [ ]:
# Delete the csv files
for f in DATA_FOLDER.glob("*.csv"):
    f.unlink()